# VQC system-level evaluation — full experiment pipeline (Kaggle)

**Settings (right panel):** Internet **On** (needed for pip). Accelerator: **None** is enough, because nothing here uses the GPU; you get the same 4 CPU cores either way.

**Recommended:** click **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 h, so the session can't time out while you're away. Expect about 2–3 h on 4 cores. Outputs appear under *Output* as `vqc_outputs.zip`.

The pinned stack (qiskit 0.45.3, numpy 1.26, scikit-learn 1.3.2) runs in its own Python 3.11 environment made with `uv`. This avoids conflicts with Kaggle's preinstalled packages and needs no kernel restart.

In [ ]:
import os, subprocess
WORK = "/kaggle/working/vqc"
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
!pip install -q uv
!uv venv -q --python 3.11 /kaggle/working/venv311
!uv pip install -q --python /kaggle/working/venv311/bin/python numpy==1.26.4 scipy==1.13.1 pandas==2.2.3 scikit-learn==1.3.2 xgboost==2.0.3 qiskit==0.45.3 qiskit-aer==0.13.3 matplotlib==3.9.4
PY = "/kaggle/working/venv311/bin/python"
os.environ["MPLBACKEND"] = "Agg"   # Kaggle sets an inline backend the venv cannot load
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["OPENBLAS_NUM_THREADS"] = "1"
N_JOBS = os.cpu_count()
print("cores:", N_JOBS)
!{PY} -c "import sys,numpy,sklearn,qiskit,qiskit_aer; print(sys.version.split()[0], numpy.__version__, sklearn.__version__, qiskit.__version__, qiskit_aer.__version__)"


## Source files
Each cell writes one module into the working directory.

In [ ]:
%%writefile common.py
"""Datasets, stratified splits and leakage-free preprocessing shared by all experiments."""
import numpy as np
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

SEEDS = list(range(42, 57))          # 15 independent stratified train/test splits
CONFIGS = [(4, 2), (8, 2), (8, 4)]    # (n qubits, L layers)
DATASETS = ["breast_cancer", "digits"]
LABEL = {"breast_cancer": "Breast Cancer", "digits": "Binary digits"}
EPS_1Q = 1e-3
EPS_2Q_GRID = [1e-3, 1e-2, 1e-1]
EPS_2Q_PRIMARY = 1e-2
SHOTS = 1024
N_ITER = 20


def load(name):
    """Return X (raw features) and y in {-1, +1}."""
    if name == "breast_cancer":
        d = load_breast_cancer()          # 569 x 30; target 0 = malignant, 1 = benign
        X, t = d.data, d.target
    elif name == "digits":
        d = load_digits()                 # 8x8 images, 64 features
        mask = d.target <= 1              # digits 0 and 1 -> 178 + 182 = 360 samples
        X, t = d.data[mask], d.target[mask]
    else:
        raise ValueError(name)
    return X.astype(float), np.where(t == 1, 1, -1)


def split(X, y, seed):
    """Stratified 80/20 split; identical for the VQC and every classical model."""
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)


def preprocess(Xtr, Xte, dim):
    """Min-max -> PCA(dim) -> min-max to [0, 1]; every step fitted on the training partition only.
    Test features are clipped to [0, 1] so that R_Y(pi x) stays in the encoding range."""
    s1 = MinMaxScaler().fit(Xtr)
    pca = PCA(n_components=dim, random_state=0).fit(s1.transform(Xtr))
    Ztr, Zte = pca.transform(s1.transform(Xtr)), pca.transform(s1.transform(Xte))
    s2 = MinMaxScaler().fit(Ztr)
    return s2.transform(Ztr), np.clip(s2.transform(Zte), 0.0, 1.0)


def matched_data(name, seed, dim):
    X, y = load(name)
    Xtr, Xte, ytr, yte = split(X, y, seed)
    Ftr, Fte = preprocess(Xtr, Xte, dim)
    return Ftr, Fte, ytr, yte


In [ ]:
%%writefile vqc_core.py
"""Core of the primary VQC simulation (Algorithm 1 of the manuscript).

Circuit (Schrodinger order), n qubits, L layers:
    R_Y(pi x_q)                         q = 0..n-1   data encoding (not trainable)
    R_Y(theta_init_q)                   q = 0..n-1   initial trainable layer
    for r = 1..L:
        R_Y(theta_Y_{q,r}) R_Z(theta_Z_{q,r})   q = 0..n-1
        CNOT(q, q+1)                            q = 0..n-2   open nearest-neighbour chain
    readout: <Z_0>

Noise: every one-qubit gate is followed by a one-qubit depolarizing channel with
parameter eps_1q and every CNOT by a two-qubit depolarizing channel with parameter
eps_2q, using the Qiskit convention E(rho) = (1-p) rho + p Tr(rho) I/d. Under this
convention every non-identity Pauli string on the channel's support is scaled by (1-p).

Expectation values are computed exactly in the Heisenberg picture: the observable Z_0
is expanded in the n-qubit Pauli basis (a (4,)*n coefficient tensor) and propagated
backwards through the trainable part of the circuit with the Pauli transfer matrices of
each gate and the adjoint (= self) depolarizing channels. The data-dependent encoding
layer acting on |0> produces a product state with Bloch vector
(1-eps_1q)(sin(pi x), 0, cos(pi x)) on each qubit, so the expectation of the propagated
observable is a product contraction that is vectorised over samples.

Finite-shot readout: N_0 ~ Binomial(S, (1+m)/2) and z_hat = (2 N_0 - S)/S (Algorithm 1).
"""
from __future__ import annotations

import numpy as np

# ------------------------------------------------------------------ Pauli algebra
_I = np.eye(2, dtype=complex)
_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = [_I, _X, _Y, _Z]  # index order I, X, Y, Z


def _heis_ptm_1q(U):
    """M[P, Q] with U^dag P U = sum_Q M[P,Q] Q."""
    M = np.empty((4, 4))
    for a, P in enumerate(PAULIS):
        O = U.conj().T @ P @ U
        for b, Q in enumerate(PAULIS):
            M[a, b] = np.real(np.trace(Q @ O)) / 2
    return M


def _heis_ptm_2q(U):
    M = np.empty((4, 4, 4, 4))
    for a, Pa in enumerate(PAULIS):
        for b, Pb in enumerate(PAULIS):
            O = U.conj().T @ np.kron(Pa, Pb) @ U
            for c, Qa in enumerate(PAULIS):
                for d, Qb in enumerate(PAULIS):
                    M[a, b, c, d] = np.real(np.trace(np.kron(Qa, Qb) @ O)) / 4
    return M


def ry(t):
    c, s = np.cos(t / 2), np.sin(t / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)


def rz(t):
    return np.array([[np.exp(-1j * t / 2), 0], [0, np.exp(1j * t / 2)]], dtype=complex)


# qubit q is the control (first tensor factor), q+1 the target
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)
M_CNOT = _heis_ptm_2q(CNOT)

# Closed forms (checked against _heis_ptm_1q in the tests)
def m_ry(t):
    c, s = np.cos(t), np.sin(t)
    # U^dag X U = c X + s Z ; U^dag Z U = -s X + c Z ; Y invariant
    return np.array([[1, 0, 0, 0], [0, c, 0, s], [0, 0, 1, 0], [0, -s, 0, c]])


def m_rz(t):
    c, s = np.cos(t), np.sin(t)
    # U^dag X U = c X - s Y ; U^dag Y U = s X + c Y ; Z invariant
    return np.array([[1, 0, 0, 0], [0, c, -s, 0], [0, s, c, 0], [0, 0, 0, 1]])


def n_params(n, L):
    return n * (2 * L + 1)


def unpack(theta, n, L):
    theta = np.asarray(theta, float)
    init = theta[:n]
    rest = theta[n:].reshape(L, 2, n)  # [layer, (Y,Z), qubit]
    return init, rest[:, 0, :], rest[:, 1, :]


def gate_list(theta, n, L):
    """Trainable gates in Schrodinger order: ('1q', q, M) or ('cx', q, None)."""
    init, ty, tz = unpack(theta, n, L)
    g = [("1q", q, m_ry(init[q])) for q in range(n)]
    for r in range(L):
        for q in range(n):
            g.append(("1q", q, m_ry(ty[r, q])))
            g.append(("1q", q, m_rz(tz[r, q])))
        for q in range(n - 1):
            g.append(("cx", q, None))
    return g




def _dep_mask_1q(n, q, lam):
    s = np.ones(4)
    s[1:] = lam
    shape = [1] * n
    shape[q] = 4
    return s.reshape(shape)


def _dep_mask_2q(n, q, lam):
    s = np.full((4, 4), lam)
    s[0, 0] = 1.0
    shape = [1] * n
    shape[q] = 4
    shape[q + 1] = 4
    return s.reshape(shape)


def propagate(theta, n, L, eps1, eps2):
    """Heisenberg-propagate Z_0 backwards through the noisy trainable circuit."""
    c = np.zeros((4,) * n)
    idx = [0] * n
    idx[0] = 3
    c[tuple(idx)] = 1.0
    lam1, lam2 = 1.0 - eps1, 1.0 - eps2
    masks1 = [_dep_mask_1q(n, q, lam1) for q in range(n)]
    masks2 = [_dep_mask_2q(n, q, lam2) for q in range(n - 1)]
    for kind, q, M in reversed(gate_list(theta, n, L)):
        if kind == "1q":
            c = c * masks1[q]                       # adjoint noise channel
            c = np.moveaxis(np.tensordot(c, M, axes=([q], [0])), -1, q)  # U^dag . U
        else:
            c = c * masks2[q]
            c = np.tensordot(c, M_CNOT, axes=([q, q + 1], [0, 1]))
            c = np.moveaxis(c, [-2, -1], [q, q + 1])
    return c


def expectation_from_observable(c, X, eps1):
    """<O> for each row of X on the noisy encoded product state."""
    lam = 1.0 - eps1
    N, n = X.shape
    a = np.pi * X
    V = np.stack([np.ones_like(a), lam * np.sin(a), np.zeros_like(a), lam * np.cos(a)], axis=-1)  # N,n,4
    # contract qubit 0 first: result (N, 4, ..., 4) with n-1 trailing axes
    R = np.tensordot(V[:, 0, :], c, axes=([1], [0]))
    for q in range(1, n):
        R = np.einsum("nj...,nj->n...", R, V[:, q, :])
    return R


def expectations(theta, X, n, L, eps1, eps2):
    return expectation_from_observable(propagate(theta, n, L, eps1, eps2), X, eps1)


def shots(m, S, rng):
    p = np.clip((1.0 + m) / 2.0, 0.0, 1.0)
    N0 = rng.binomial(S, p)
    return (2.0 * N0 - S) / S


def predict_labels(zhat):
    return np.where(zhat >= 0, 1, -1)


def spsa_train(X, y, n, L, eps1, eps2, seed, n_iter=20, S=1024,
               a=0.05, c=0.10, A=10, alpha=0.602, gamma=0.101, history=False):
    """Algorithm 1. Returns theta*, rng (to continue the stream), and optional loss history."""
    rng = np.random.default_rng(seed)
    P = n_params(n, L)
    theta = rng.uniform(-np.pi, np.pi, P)
    hist = []
    for k in range(n_iter):
        ak = a / (A + k + 1) ** alpha
        ck = c / (k + 1) ** gamma
        delta = rng.choice([-1.0, 1.0], size=P)
        losses = {}
        for s in (+1, -1):
            th = theta + s * ck * delta
            m = expectations(th, X, n, L, eps1, eps2)
            z = shots(m, S, rng)
            losses[s] = float(np.mean((z - y) ** 2))
        g = (losses[+1] - losses[-1]) / (2 * ck) * delta  # Delta^{-1} = Delta for +-1 entries
        theta = theta - ak * g
        if history:
            hist.append(0.5 * (losses[+1] + losses[-1]))
    return theta, rng, hist


def evaluate(theta, X, y, n, L, eps1, eps2, rng, S=1024):
    m = expectations(theta, X, n, L, eps1, eps2)
    z = shots(m, S, rng)
    return float(np.mean(predict_labels(z) == y)), m


In [ ]:
%%writefile test_vqc_core.py
"""Validate the Pauli-propagation simulator against Qiskit Aer density-matrix simulation."""
import time
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import vqc_core as V


def qiskit_circuit(x, theta, n, L):
    init, ty, tz = V.unpack(theta, n, L)
    qc = QuantumCircuit(n)
    for q in range(n): qc.ry(np.pi * x[q], q)
    for q in range(n): qc.ry(init[q], q)
    for r in range(L):
        for q in range(n):
            qc.ry(ty[r, q], q); qc.rz(tz[r, q], q)
        for q in range(n - 1): qc.cx(q, q + 1)
    return qc


def aer_expectation(x, theta, n, L, e1, e2):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(e1, 1), ["ry", "rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(e2, 2), ["cx"])
    qc = qiskit_circuit(x, theta, n, L)
    qc.save_density_matrix()
    sim = AerSimulator(method="density_matrix", noise_model=nm)
    rho = sim.run(qc).result().data()["density_matrix"]
    return float(np.real(rho.expectation_value(SparsePauliOp("I" * (n - 1) + "Z"))))


# closed-form PTMs equal the numerically derived ones
for t in np.linspace(-3, 3, 7):
    assert np.allclose(V.m_ry(t), V._heis_ptm_1q(V.ry(t)))
    assert np.allclose(V.m_rz(t), V._heis_ptm_1q(V.rz(t)))
print("PTM closed forms OK")

rng = np.random.default_rng(0)
worst = 0
for n, L in [(4, 2), (8, 2), (8, 4)]:
    for e1, e2 in [(0, 0), (1e-3, 1e-2), (1e-3, 1e-1), (5e-2, 2e-1)]:
        th = rng.uniform(-np.pi, np.pi, V.n_params(n, L))
        X = rng.uniform(0, 1, (5, n))
        mine = V.expectations(th, X, n, L, e1, e2)
        ref = np.array([aer_expectation(x, th, n, L, e1, e2) for x in X])
        err = np.max(np.abs(mine - ref)); worst = max(worst, err)
        print(f"n={n} L={L} eps=({e1},{e2})  max|diff|={err:.2e}   e.g. {mine[0]:+.6f} vs {ref[0]:+.6f}")
assert worst < 1e-9, worst
t = time.perf_counter(); V.expectations(th, rng.uniform(0, 1, (455, 8)), 8, 4, 1e-3, 1e-2)
print(f"ALL MATCH (worst {worst:.1e}); 8Q-4L, 455 samples: {time.perf_counter()-t:.3f} s")


In [ ]:
%%writefile run_vqc.py
"""Primary VQC experiment: 2 datasets x 3 configurations x 15 splits.

For every (dataset, n, L, seed):
  * retrained protocol: Algorithm 1 is run separately at eps_2q in {1e-3, 1e-2, 1e-1}
    and the trained model is evaluated on the test partition at the same noise level;
  * fixed-model protocol: the parameters trained at eps_2q = 1e-2 are re-evaluated,
    without retraining, at all three eps_2q values.
eps_1q = 1e-3 throughout. Output: results/vqc_trials.csv (one row per evaluation) and
results/vqc_loss_history.csv.
"""
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import common as C
import vqc_core as V

OUT = Path(sys.argv[1] if len(sys.argv) > 1 else "results")
OUT.mkdir(parents=True, exist_ok=True)

rows, hist_rows = [], []
t_all = time.perf_counter()
for ds in C.DATASETS:
    for n, L in C.CONFIGS:
        for seed in C.SEEDS:
            Ftr, Fte, ytr, yte = C.matched_data(ds, seed, n)
            trained = {}
            for e2 in C.EPS_2Q_GRID:
                t0 = time.perf_counter()
                theta, rng, hist = V.spsa_train(Ftr, ytr, n, L, C.EPS_1Q, e2, seed,
                                                n_iter=C.N_ITER, S=C.SHOTS, history=True)
                t_train = time.perf_counter() - t0
                acc_tr, _ = V.evaluate(theta, Ftr, ytr, n, L, C.EPS_1Q, e2, rng, C.SHOTS)
                t0 = time.perf_counter()
                acc_te, m = V.evaluate(theta, Fte, yte, n, L, C.EPS_1Q, e2, rng, C.SHOTS)
                t_eval = time.perf_counter() - t0
                acc_exact = float(np.mean(V.predict_labels(m) == yte))
                trained[e2] = theta
                rows.append(dict(dataset=ds, n_qubits=n, n_layers=L, seed=seed, protocol="retrained",
                                 eps_2q_train=e2, eps_2q=e2, test_accuracy=acc_te,
                                 test_accuracy_noiseless_readout=acc_exact, train_accuracy=acc_tr,
                                 n_train=len(ytr), n_test=len(yte), train_seconds=t_train,
                                 eval_seconds=t_eval))
                for k, l in enumerate(hist):
                    hist_rows.append(dict(dataset=ds, n_qubits=n, n_layers=L, seed=seed, eps_2q=e2,
                                          iteration=k, loss=l))
            theta = trained[C.EPS_2Q_PRIMARY]
            for j, e2 in enumerate(C.EPS_2Q_GRID):
                rng = np.random.default_rng([seed, 7, j])
                acc_te, _ = V.evaluate(theta, Fte, yte, n, L, C.EPS_1Q, e2, rng, C.SHOTS)
                rows.append(dict(dataset=ds, n_qubits=n, n_layers=L, seed=seed, protocol="fixed_model",
                                 eps_2q_train=C.EPS_2Q_PRIMARY, eps_2q=e2, test_accuracy=acc_te,
                                 n_train=len(ytr), n_test=len(yte)))
            r = [x for x in rows if x["protocol"] == "retrained" and x["eps_2q"] == 1e-2][-1]
            print(f"{ds:13s} {n}Q-{L}L seed {seed}: test acc @1e-2 = {r['test_accuracy']:.4f} "
                  f"(train {r['train_accuracy']:.3f}, {r['train_seconds']:.1f}s)  [{time.perf_counter()-t_all:.0f}s]",
                  flush=True)
        pd.DataFrame(rows).to_csv(OUT / "vqc_trials.csv", index=False)
        pd.DataFrame(hist_rows).to_csv(OUT / "vqc_loss_history.csv", index=False)
print("done", time.perf_counter() - t_all)


In [ ]:
%%writefile run_classical.py
"""Classical baselines.

mode 'matched' (primary, Table 6): the six model families on exactly the same 15
stratified splits and the same leakage-free PCA features (dimension 4 and 8) as the VQC.
mode 'nested' (secondary, Table 5): 5 repetitions x 5-fold outer CV on the full
min-max-scaled feature set, with hyperparameters chosen by inner 5-fold CV.

Logistic regression (l2, C=1) and linear SVM (C=1) are fixed; RBF SVM, random forest, MLP
and XGBoost are grid-searched (inner stratified 5-fold CV on the training partition only)
over the grids stated in the manuscript. scikit-learn's MLPClassifier has no dropout, so
the MLP grid is capacity x learning rate only.
"""
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

import common as C

warnings.filterwarnings("ignore", category=ConvergenceWarning)
MODE = sys.argv[1]
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "results"); OUT.mkdir(parents=True, exist_ok=True)
N_JOBS = int(sys.argv[3]) if len(sys.argv) > 3 else 2

GRIDS = {
    "RBF SVM": (SVC(kernel="rbf"), {"C": [0.1, 1, 10, 100], "gamma": ["scale", "auto", 1e-3, 1e-2]}),
    "Random Forest": (RandomForestClassifier(n_jobs=1, random_state=0),
                      {"n_estimators": [100, 200, 500], "max_depth": [5, 10, 20, None], "min_samples_leaf": [1, 2, 4]}),
    "MLP": (MLPClassifier(max_iter=1000, random_state=0),
            {"hidden_layer_sizes": [(64, 64), (128, 128), (256, 256, 128)], "learning_rate_init": [1e-4, 1e-3, 1e-2]}),
    "XGBoost": (XGBClassifier(n_jobs=1, random_state=0, tree_method="hist", eval_metric="logloss"),
                {"n_estimators": [100, 300, 500], "max_depth": [3, 6, 9], "learning_rate": [0.01, 0.05, 0.1, 0.2],
                 "subsample": [0.7, 0.9, 1.0]}),
}
ORDER = ["Logistic Regression", "Linear SVM", "RBF SVM", "Random Forest", "MLP", "XGBoost"]


def fit_predict(name, Xtr, ytr, Xte, seed):
    y01 = (ytr + 1) // 2
    if name == "Logistic Regression":
        m = LogisticRegression(C=1.0, penalty="l2", max_iter=5000).fit(Xtr, y01); best = {}
    elif name == "Linear SVM":
        m = SVC(kernel="linear", C=1.0).fit(Xtr, y01); best = {}
    else:
        est, grid = GRIDS[name]
        gs = GridSearchCV(est, grid, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          scoring="accuracy", n_jobs=N_JOBS, refit=True).fit(Xtr, y01)
        m, best = gs.best_estimator_, gs.best_params_
    return 2 * m.predict(Xte) - 1, best


rows = []
t_all = time.perf_counter()
_csv = OUT / ("classical_matched_trials.csv" if MODE == "matched" else "classical_nested_cv.csv")
if _csv.exists():                     # resume: keep completed units, skip them below
    rows = pd.read_csv(_csv).to_dict("records")
done_m = {(r["dataset"], r["seed"]) for r in rows} if MODE == "matched" else set()
done_n = {(r["dataset"], r["repetition"]) for r in rows} if MODE == "nested" else set()
if MODE == "matched":
    for ds in C.DATASETS:
        for seed in C.SEEDS:
            if (ds, seed) in done_m and sum(1 for r in rows if r["dataset"] == ds and r["seed"] == seed) == 12:
                continue
            rows = [r for r in rows if not (r["dataset"] == ds and r["seed"] == seed)]
            for dim in (4, 8):
                Ftr, Fte, ytr, yte = C.matched_data(ds, seed, dim)
                for name in ORDER:
                    t0 = time.perf_counter()
                    pred, best = fit_predict(name, Ftr, ytr, Fte, seed)
                    rows.append(dict(dataset=ds, model=name, dimension=dim, seed=seed,
                                     test_accuracy=float(np.mean(pred == yte)), n_test=len(yte),
                                     fit_seconds=time.perf_counter() - t0, best_params=str(best)))
                print(f"{ds} seed {seed} dim {dim}: " + ", ".join(f"{r['model'][:6]}={r['test_accuracy']:.3f}" for r in rows[-6:])
                      + f"  [{time.perf_counter()-t_all:.0f}s]", flush=True)
            pd.DataFrame(rows).to_csv(OUT / "classical_matched_trials.csv", index=False)
elif MODE == "nested":
    for ds in C.DATASETS:
        X, y = C.load(ds)
        for rep in range(5):
            if (ds, rep) in done_n and sum(1 for r in rows if r["dataset"] == ds and r["repetition"] == rep) == 30:
                continue
            rows = [r for r in rows if not (r["dataset"] == ds and r["repetition"] == rep)]
            outer = StratifiedKFold(5, shuffle=True, random_state=rep)
            for fold, (tr, te) in enumerate(outer.split(X, y)):
                sc = MinMaxScaler().fit(X[tr])
                Xtr, Xte = sc.transform(X[tr]), np.clip(sc.transform(X[te]), 0, 1)
                for name in ORDER:
                    pred, best = fit_predict(name, Xtr, y[tr], Xte, 100 * rep + fold)
                    rows.append(dict(dataset=ds, model=name, repetition=rep, fold=fold,
                                     test_accuracy=float(np.mean(pred == y[te])), best_params=str(best)))
                print(f"{ds} rep {rep} fold {fold}: " + ", ".join(f"{r['model'][:6]}={r['test_accuracy']:.3f}" for r in rows[-6:])
                      + f"  [{time.perf_counter()-t_all:.0f}s]", flush=True)
            pd.DataFrame(rows).to_csv(OUT / "classical_nested_cv.csv", index=False)
print("done", time.perf_counter() - t_all)


In [ ]:
%%writefile run_aer.py
"""Qiskit Aer experiments on the final circuit architecture.

mode 'timing'   (Table 9, Figure 8): for each configuration, 20 circuits (the trained
    seed-42 Breast Cancer parameters bound to the first 20 test samples) are transpiled
    (basis {rz, sx, x, cx}, linear coupling map, optimization_level=2, SABRE layout and
    routing, seed_transpiler=42) and executed on AerSimulator with depolarizing noise
    (eps_1q = 1e-3 on rz/sx/x, eps_2q = 1e-2 on cx) and 1024 shots. Compile and execution
    wall-clock times are measured separately; 1 warm-up + 10 timed repetitions.
mode 'nairobi'  (Table 10, Figure 9): Breast Cancer, n = 4, L = 2, the same 15 splits (seeds 42-56). Algorithm 1
    is run with every loss evaluation estimated from 1024-shot counts on
    AerSimulator.from_backend(FakeNairobiV2()) (calibrated gate, readout and thermal-relaxation
    noise of the offline snapshot); the test partition is evaluated the same way.
"""
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

import common as C
import vqc_core as V

MODE = sys.argv[1]
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "results"); OUT.mkdir(parents=True, exist_ok=True)


def param_circuit(n, L):
    x = ParameterVector("x", n)
    th = ParameterVector("t", V.n_params(n, L))
    qc = QuantumCircuit(n, 1)
    for q in range(n): qc.ry(np.pi * x[q], q)
    for q in range(n): qc.ry(th[q], q)
    k = n
    for r in range(L):
        for q in range(n): qc.ry(th[k + q], q)
        for q in range(n): qc.rz(th[k + n + q], q)
        k += 2 * n
        for q in range(n - 1): qc.cx(q, q + 1)
    qc.measure(0, 0)
    return qc, x, th


def z_from_counts(counts, S):
    n0 = counts.get("0", 0)
    return (2 * n0 - S) / S


if MODE == "timing":
    trials = pd.read_csv(OUT / "vqc_trials.csv")
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(C.EPS_1Q, 1), ["rz", "sx", "x"])
    nm.add_all_qubit_quantum_error(depolarizing_error(C.EPS_2Q_PRIMARY, 2), ["cx"])
    rows, meta = [], {}
    for n, L in C.CONFIGS:
        Ftr, Fte, ytr, yte = C.matched_data("breast_cancer", 42, n)
        theta, _, _ = V.spsa_train(Ftr, ytr, n, L, C.EPS_1Q, C.EPS_2Q_PRIMARY, 42, n_iter=C.N_ITER, S=C.SHOTS)
        qc, xp, tp = param_circuit(n, L)
        circs = [qc.assign_parameters({**dict(zip(xp, Fte[i])), **dict(zip(tp, theta))}) for i in range(20)]
        cmap = CouplingMap.from_line(n)
        sim = AerSimulator(noise_model=nm)
        kw = dict(basis_gates=["rz", "sx", "x", "cx"], coupling_map=cmap, optimization_level=2,
                  layout_method="sabre", routing_method="sabre", seed_transpiler=42)
        tq = transpile(circs, **kw); sim.run(tq, shots=C.SHOTS, seed_simulator=0).result()  # warm-up
        for rep in range(10):
            t0 = time.perf_counter(); tq = transpile(circs, **kw); tc = time.perf_counter() - t0
            t0 = time.perf_counter(); res = sim.run(tq, shots=C.SHOTS, seed_simulator=rep).result(); te = time.perf_counter() - t0
            rows.append(dict(n_qubits=n, n_layers=L, repetition=rep, compile_s=tc, execute_s=te))
        ops = [dict(c.count_ops()) for c in tq]
        meta[f"{n}Q-{L}L"] = dict(method=res.results[0].metadata.get("method"),
                                  transpiled_depth_mean=float(np.mean([c.depth() for c in tq])),
                                  transpiled_cx_mean=float(np.mean([o.get("cx", 0) for o in ops])),
                                  transpiled_1q_mean=float(np.mean([sum(v for k, v in o.items() if k in ("rz", "sx", "x")) for o in ops])),
                                  logical_counts=dict(qc.count_ops()))
        d = pd.DataFrame(rows[-10:])
        print(f"{n}Q-{L}L compile {d.compile_s.mean():.4f}±{d.compile_s.std():.4f}  exec {d.execute_s.mean():.4f}±{d.execute_s.std():.4f}  {meta[f'{n}Q-{L}L']}", flush=True)
    pd.DataFrame(rows).to_csv(OUT / "aer_timing.csv", index=False)
    (OUT / "aer_timing_meta.json").write_text(json.dumps(meta, indent=2))

elif MODE == "nairobi":
    from qiskit.providers.fake_provider import FakeNairobiV2
    backend = FakeNairobiV2()
    sim = AerSimulator.from_backend(backend)
    n, L, S = 4, 2, C.SHOTS
    qc, xp, tp = param_circuit(n, L)
    tqc = transpile(qc, backend=backend, optimization_level=2, layout_method="sabre",
                    routing_method="sabre", seed_transpiler=42)
    layout = tqc.layout.final_index_layout() if tqc.layout is not None else None
    rows = []
    for seed in C.SEEDS:
        Ftr, Fte, ytr, yte = C.matched_data("breast_cancer", seed, n)
        rng = np.random.default_rng(seed)
        P = V.n_params(n, L)
        theta = rng.uniform(-np.pi, np.pi, P)
        sim_seed = [int(seed) * 1000]

        def zhat(th, X):
            binds = {p: list(X[:, i]) for i, p in enumerate(xp)}
            binds.update({p: [float(th[i])] * len(X) for i, p in enumerate(tp)})
            sim_seed[0] += 1
            res = sim.run(tqc, shots=S, parameter_binds=[binds], seed_simulator=sim_seed[0]).result()
            return np.array([z_from_counts(res.get_counts(i), S) for i in range(len(X))])

        t0 = time.perf_counter()
        for k in range(C.N_ITER):
            ak = 0.05 / (10 + k + 1) ** 0.602; ck = 0.10 / (k + 1) ** 0.101
            delta = rng.choice([-1.0, 1.0], size=P)
            lp = np.mean((zhat(theta + ck * delta, Ftr) - ytr) ** 2)
            lm = np.mean((zhat(theta - ck * delta, Ftr) - ytr) ** 2)
            theta = theta - ak * (lp - lm) / (2 * ck) * delta
        t_train = time.perf_counter() - t0
        t0 = time.perf_counter(); z = zhat(theta, Fte); t_eval = time.perf_counter() - t0
        acc = float(np.mean(V.predict_labels(z) == yte))
        # same trained parameters under the synthetic Pauli-propagation model, for reference
        acc_syn, _ = V.evaluate(theta, Fte, yte, n, L, C.EPS_1Q, C.EPS_2Q_PRIMARY, np.random.default_rng([seed, 9]), S)
        rows.append(dict(seed=seed, test_accuracy=acc, test_accuracy_same_params_synthetic=acc_syn,
                         n_train=len(ytr), n_test=len(yte), train_seconds=t_train, eval_seconds=t_eval,
                         physical_layout=str(layout), backend=backend.name))
        print(rows[-1], flush=True)
    pd.DataFrame(rows).to_csv(OUT / "fakenairobi_trials.csv", index=False)


In [ ]:
%%writefile build_results.py
"""Build every reported table and data figure from the trial-level CSVs in results/.

Statistics: mean; sample SD (ddof = 1); percentile bootstrap 95% CI of the mean
(B = 10,000 resamples, numpy default_rng(2026)). Paired comparisons (same split seed):
mean paired difference with percentile bootstrap CI, paired t-test and exact Wilcoxon
signed-rank test, Holm correction over the six VQC-vs-reference comparisons.
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import common as C

RES = Path(sys.argv[1] if len(sys.argv) > 1 else "results")
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "tables"); OUT.mkdir(parents=True, exist_ok=True)
FIG = Path(sys.argv[3] if len(sys.argv) > 3 else "figures"); FIG.mkdir(parents=True, exist_ok=True)
B, BOOT_SEED = 10_000, 2026
rng = np.random.default_rng(BOOT_SEED)
ORDER = ["Logistic Regression", "Linear SVM", "RBF SVM", "Random Forest", "MLP", "XGBoost"]
COL = {(4, 2): "#2a78d6", (8, 2): "#eb6834", (8, 4): "#1baf7a"}
MK = {(4, 2): "o", (8, 2): "s", (8, 4): "D"}
TAG = {c: f"{c[0]}Q\u2013{c[1]}L" for c in C.CONFIGS}
plt.rcParams.update({"font.size": 10, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
                     "grid.color": "#dddddd", "grid.linewidth": 0.6})


def g1q(n, L): return n * (2 * L + 2)
def g2q(n, L): return (n - 1) * L
def burden(n, L): return g1q(n, L) + 2 * g2q(n, L)
def fid(n, L, e2=1e-2): return (1 - C.EPS_1Q) ** g1q(n, L) * (1 - e2) ** g2q(n, L)


def boot(x):
    x = np.asarray(x, float)
    m = x[rng.integers(0, len(x), (B, len(x)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


def summ(x):
    x = np.asarray(x, float); lo, hi = boot(x)
    return dict(n=len(x), mean=float(x.mean()), sd=float(x.std(ddof=1)), lo=lo, hi=hi)


def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj


S = {"bootstrap": dict(B=B, seed=BOOT_SEED)}
vqc = pd.read_csv(RES / "vqc_trials.csv")
cls = pd.read_csv(RES / "classical_matched_trials.csv")
ret = vqc[vqc.protocol == "retrained"]; fix = vqc[vqc.protocol == "fixed_model"]

# integrity: identical seeds everywhere
for ds in C.DATASETS:
    for (n, L) in C.CONFIGS:
        s = set(ret[(ret.dataset == ds) & (ret.n_qubits == n) & (ret.n_layers == L)].seed)
        assert s == set(C.SEEDS), (ds, n, L, s)
    for (m, d), g in cls[cls.dataset == ds].groupby(["model", "dimension"]):
        assert set(g.seed) == set(C.SEEDS), (ds, m, d)

# ---------------------------------------------------------------- majority-class reference
S["majority"] = {}
for ds in C.DATASETS:
    acc = []
    for seed in C.SEEDS:
        _, _, ytr, yte = C.matched_data(ds, seed, 4)
        maj = 1 if (ytr == 1).sum() >= (ytr == -1).sum() else -1
        acc.append(float(np.mean(yte == maj)))
    S["majority"][ds] = summ(acc)

# ---------------------------------------------------------------- Table 7
S["vqc"] = {}; rows = []
for ds in C.DATASETS:
    for n, L in C.CONFIGS:
        x = ret[(ret.dataset == ds) & (ret.n_qubits == n) & (ret.n_layers == L) & np.isclose(ret.eps_2q, 1e-2)].sort_values("seed")
        s = summ(x.test_accuracy); s["train_mean"] = float(x.train_accuracy.mean())
        s["min"], s["max"] = float(x.test_accuracy.min()), float(x.test_accuracy.max())
        s["below_half"] = int((x.test_accuracy < 0.5).sum())
        S["vqc"][f"{ds}|{n}|{L}"] = s
        rows.append(f"{C.LABEL[ds]} & {n} & {L} & {s['mean']:.4f} & {s['sd']:.4f} & [{s['lo']:.4f}, {s['hi']:.4f}] \\\\")
(OUT / "table7_vqc.tex").write_text("\n".join(rows) + "\n")

# ---------------------------------------------------------------- Table 6 + A_ref
S["classical"] = {}; S["A_ref"] = {}; blocks = []
for ds in C.DATASETS:
    sub = cls[cls.dataset == ds]; lines = []; best = (None, -1)
    for d in (4, 8):
        for m in ORDER:
            s = summ(sub[(sub.model == m) & (sub.dimension == d)].sort_values("seed").test_accuracy)
            S["classical"][f"{ds}|{m}|{d}"] = s
            lines.append(f"& {m:19s} & {d} & {s['mean']:.4f} & {s['sd']:.4f} & [{s['lo']:.4f}, {s['hi']:.4f}] \\\\")
            if s["mean"] > best[1] + 1e-12: best = ((m, d), s["mean"])
    S["A_ref"][ds] = dict(model=best[0][0], dimension=best[0][1], mean=best[1])
    blocks.append(f"\\multirow{{12}}{{*}}{{{C.LABEL[ds]}}}\n" + "\n".join(lines))
(OUT / "table6_matched.tex").write_text("\n\n\\midrule\n\n".join(blocks) + "\n")
S["vqc_ci_max_upper"] = max(v["hi"] for v in S["vqc"].values())
S["classical_ci_min_lower"] = min(v["lo"] for v in S["classical"].values())

# ---------------------------------------------------------------- paired tests
pr = []
for ds in C.DATASETS:
    R = S["A_ref"][ds]
    ref = cls[(cls.dataset == ds) & (cls.model == R["model"]) & (cls.dimension == R["dimension"])].set_index("seed").test_accuracy
    for n, L in C.CONFIGS:
        # also the best classical model at the VQC's own dimension
        v = ret[(ret.dataset == ds) & (ret.n_qubits == n) & (ret.n_layers == L) & np.isclose(ret.eps_2q, 1e-2)].set_index("seed").test_accuracy
        d = (v.loc[C.SEEDS] - ref.loc[C.SEEDS]).to_numpy()
        lo, hi = boot(d)
        t = stats.ttest_rel(v.loc[C.SEEDS], ref.loc[C.SEEDS])
        w = stats.wilcoxon(d, zero_method="wilcox", method="exact" if np.all(d != 0) else "approx")
        pr.append(dict(dataset=ds, n=n, L=L, reference=f"{R['model']} ({R['dimension']}D)", mean_diff=float(d.mean()),
                       lo=lo, hi=hi, wins=int((d > 0).sum()), t=float(t.statistic), p_t=float(t.pvalue),
                       W=float(w.statistic), p_w=float(w.pvalue)))
pt = pd.DataFrame(pr); pt["p_t_holm"] = holm(pt.p_t); pt["p_w_holm"] = holm(pt.p_w)
pt.to_csv(OUT / "paired_tests.csv", index=False); S["paired"] = pt.to_dict("records")


def fmt_p(p):
    return f"{p:.1e}".replace("e-0", "\\times10^{-").replace("e-", "\\times10^{-") + "}" if p < 1e-3 else f"{p:.3f}"


(OUT / "table_paired.tex").write_text("\n".join(
    f"{C.LABEL[r.dataset]} & {r.n} & {r.L} & {r.reference} & ${r.mean_diff:.4f}$ & [${r.lo:.4f}$, ${r.hi:.4f}$] & {r.wins}/15 & ${fmt_p(r.p_t_holm)}$ & ${fmt_p(r.p_w_holm)}$ \\\\"
    for r in pt.itertuples()) + "\n")

# ---------------------------------------------------------------- Table 8 / Figure 7 (noise sensitivity)
S["noise"] = {}; rows = []
for ds in C.DATASETS:
    for n, L in C.CONFIGS:
        cells = []
        for proto, df in (("fixed", fix), ("retrained", ret)):
            for e in C.EPS_2Q_GRID:
                x = df[(df.dataset == ds) & (df.n_qubits == n) & (df.n_layers == L) & np.isclose(df.eps_2q, e)].test_accuracy
                s = summ(x); S["noise"][f"{proto}|{ds}|{n}|{L}|{e:g}"] = s
                cells.append(f"{s['mean']:.3f} $\\pm$ {s['sd']:.3f}")
        rows.append(f"{C.LABEL[ds]} & {n} & {L} & " + " & ".join(cells) + " \\\\")
(OUT / "table8_noise.tex").write_text("\n".join(rows) + "\n")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw=dict(width_ratios=[1.15, 1]))
lab = [f"{'BC' if ds == 'breast_cancer' else 'Digits'} {TAG[c]}" for ds in C.DATASETS for c in C.CONFIGS]
M = np.array([[S["noise"][f"fixed|{ds}|{c[0]}|{c[1]}|{e:g}"]["mean"] for e in C.EPS_2Q_GRID] for ds in C.DATASETS for c in C.CONFIGS])
im = ax[0].imshow(M, cmap="Blues", vmin=0.4, vmax=1.0, aspect="auto")
for i in range(M.shape[0]):
    for j in range(3):
        ax[0].text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", fontsize=9, color="white" if M[i, j] > 0.75 else "#222222")
ax[0].set_xticks(range(3), ["$10^{-3}$", "$10^{-2}$", "$10^{-1}$"]); ax[0].set_yticks(range(6), lab)
ax[0].set_xlabel("Evaluation $\\epsilon_{2q}$ (parameters trained at $10^{-2}$)")
ax[0].set_title("(a) Fixed-model mean test accuracy (15 splits)")
for s in ax[0].spines.values(): s.set_visible(False)
fig.colorbar(im, ax=ax[0], fraction=0.04, pad=0.02)
amp_f = M.max(1) - M.min(1)
Mr = np.array([[S["noise"][f"retrained|{ds}|{c[0]}|{c[1]}|{e:g}"]["mean"] for e in C.EPS_2Q_GRID] for ds in C.DATASETS for c in C.CONFIGS])
amp_r = Mr.max(1) - Mr.min(1)
y = np.arange(6)
ax[1].barh(y - 0.19, amp_f, 0.36, color="#2a78d6", label="Fixed model")
ax[1].barh(y + 0.19, amp_r, 0.36, color="#eb6834", label="Retrained per $\\epsilon_{2q}$")
for i in range(6):
    ax[1].text(amp_f[i] + 0.002, i - 0.19, f"{amp_f[i]:.3f}", va="center", fontsize=8.5)
    ax[1].text(amp_r[i] + 0.002, i + 0.19, f"{amp_r[i]:.3f}", va="center", fontsize=8.5)
ax[1].set_yticks(y, lab); ax[1].invert_yaxis(); ax[1].set_xlabel("Range of mean accuracy across $\\epsilon_{2q}$")
ax[1].set_title("(b) Sensitivity amplitude"); ax[1].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2); ax[1].grid(axis="x")
ax[1].set_xlim(0, max(amp_f.max(), amp_r.max()) * 1.25 + 0.01)
fig.tight_layout(); fig.savefig(FIG / "Figure_7_Noise_Sensitivity.png", bbox_inches="tight"); plt.close(fig)
S["noise_amplitude"] = dict(fixed=dict(zip(lab, amp_f.tolist())), retrained=dict(zip(lab, amp_r.tolist())))

# ---------------------------------------------------------------- Figure 6
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
for k, (axx, ds) in enumerate(zip(axes, C.DATASETS)):
    R = S["A_ref"][ds]
    for i, c in enumerate(C.CONFIGS):
        x = ret[(ret.dataset == ds) & (ret.n_qubits == c[0]) & (ret.n_layers == c[1]) & np.isclose(ret.eps_2q, 1e-2)].test_accuracy
        jit = np.random.default_rng(i).uniform(-0.12, 0.12, len(x))
        axx.scatter(i + jit, x, s=14, color=COL[c], alpha=0.35, linewidths=0, zorder=2)
        s = S["vqc"][f"{ds}|{c[0]}|{c[1]}"]
        axx.errorbar(i, s["mean"], yerr=[[s["mean"] - s["lo"]], [s["hi"] - s["mean"]]], fmt=MK[c], color=COL[c],
                     ms=9, capsize=6, lw=2, mec="white", mew=1.2, zorder=3)
        axx.text(i, 0.03, f"$\\Delta A$={s['mean'] - R['mean']:.3f}", ha="center", va="bottom", fontsize=9, color="#222222")
    axx.axhline(R["mean"], ls="--", color="#555555", lw=1.4)
    axx.text(-0.45, R["mean"] + 0.012, f"Strongest matched classical: {R['model']} ({R['dimension']}D) = {R['mean']:.4f}", ha="left", va="bottom", fontsize=8.5, color="#333333")
    M = S["majority"][ds]["mean"]
    axx.axhline(M, ls=":", color="#888888", lw=1.2, zorder=1)
    axx.text(0.5, M + 0.01, f"Majority class = {M:.3f}", ha="center", va="bottom", fontsize=8, color="#555555",
             bbox=dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.85), zorder=4)
    axx.set_xticks(range(3), [TAG[c] for c in C.CONFIGS]); axx.set_xlim(-0.5, 2.5); axx.set_ylim(0, 1.12)
    axx.grid(axis="y"); axx.set_xlabel("VQC configuration"); axx.set_title(f"({'ab'[k]}) {C.LABEL[ds]}")
axes[0].set_ylabel("Test accuracy ($\\epsilon_{2q}=10^{-2}$, 15 matched splits)")
fig.tight_layout(); fig.savefig(FIG / "Figure_6_Performance_Landscape.png", bbox_inches="tight"); plt.close(fig)

# ---------------------------------------------------------------- Table 5 (nested CV)
nest = pd.read_csv(RES / "classical_nested_cv.csv") if (RES / "classical_nested_cv.csv").exists() else None
S["nested"] = {}
if nest is not None:
    rows = []
    for ds in C.DATASETS:
        for j, m in enumerate(ORDER):
            g = nest[(nest.dataset == ds) & (nest.model == m)]
            rep_means = g.groupby("repetition").test_accuracy.mean()
            S["nested"][f"{ds}|{m}"] = dict(mean=float(g.test_accuracy.mean()), sd_reps=float(rep_means.std(ddof=1)),
                                            sd_folds=float(g.test_accuracy.std(ddof=1)), n_folds=len(g))
            pre = f"\\multirow{{6}}{{*}}{{{C.LABEL[ds]}}} " if j == 0 else ""
            rows.append(f"{pre}& {m} & {g.test_accuracy.mean():.4f} & {rep_means.std(ddof=1):.4f} \\\\")
        rows.append("\\midrule")
    (OUT / "table5_nested.tex").write_text("\n".join(rows[:-1]) + "\n")

# ---------------------------------------------------------------- Table 9 / Figure 8 (timing)
T = {}
if (RES / "aer_timing.csv").exists():
    tm = pd.read_csv(RES / "aer_timing.csv"); meta = json.loads((RES / "aer_timing_meta.json").read_text())
    rows = []
    for n, L in C.CONFIGS:
        g = tm[(tm.n_qubits == n) & (tm.n_layers == L)]
        tot = g.compile_s + g.execute_s; share = 100 * g.execute_s / tot
        T[(n, L)] = float(tot.mean())
        k = f"{n}Q-{L}L"
        S.setdefault("timing", {})[k] = dict(compile=float(g.compile_s.mean()), compile_sd=float(g.compile_s.std(ddof=1)),
                                             execute=float(g.execute_s.mean()), execute_sd=float(g.execute_s.std(ddof=1)),
                                             total=float(tot.mean()), share=float(share.mean()), **meta[k])
        t = S["timing"][k]
        rows.append(f"{n} & {L} & {t['compile']:.3f} $\\pm$ {t['compile_sd']:.3f} & {t['execute']:.3f} $\\pm$ {t['execute_sd']:.3f} & {t['share']:.1f} & {t['transpiled_depth_mean']:.1f} & {t['transpiled_cx_mean']:.0f} \\\\")
    (OUT / "table9_timing.tex").write_text("\n".join(rows) + "\n")
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for c in C.CONFIGS:
        t = S["timing"][f"{c[0]}Q-{c[1]}L"]
        ax.scatter(burden(*c), t["total"], s=4 * burden(*c), color=COL[c], marker=MK[c], edgecolor="white", linewidth=1.5, zorder=3)
        ax.errorbar(burden(*c), t["total"], yerr=np.hypot(t["compile_sd"], t["execute_sd"]), color="#555555", lw=1, capsize=3, zorder=2)
        ax.annotate(f"{TAG[c]} (C={burden(*c)})\n{t['total']:.3f} s, {t['share']:.1f}% execution", (burden(*c), t["total"]),
                    xytext=(16, -2) if c != (8, 4) else (-16, -22), textcoords="offset points", fontsize=8.5,
                    ha="left" if c != (8, 4) else "right", va="center")
    ax.set_xlim(20, 155); ax.set_ylim(0, max(T.values()) * 1.25)
    ax.set_xlabel("Weighted circuit burden $C=G_{1q}+2G_{2q}$"); ax.set_ylabel("Local Aer compile + execution time (s, 20 circuits)")
    ax.grid(True); fig.tight_layout(); fig.savefig(FIG / "Figure_8_Runtime_Burden.png", bbox_inches="tight"); plt.close(fig)

# ---------------------------------------------------------------- Table 10 / Figure 9 (FakeNairobiV2)
if (RES / "fakenairobi_trials.csv").exists():
    fn = pd.read_csv(RES / "fakenairobi_trials.csv").sort_values("seed")
    syn = ret[(ret.dataset == "breast_cancer") & (ret.n_qubits == 4) & (ret.n_layers == 2) & np.isclose(ret.eps_2q, 1e-2)].set_index("seed").test_accuracy
    d = fn.set_index("seed").test_accuracy - syn.loc[fn.seed]
    lo, hi = boot(d.to_numpy())
    w = stats.wilcoxon(d.to_numpy(), method="approx") if np.any(d != 0) else None
    S["nairobi"] = dict(acc=summ(fn.test_accuracy), acc_min=float(fn.test_accuracy.min()), acc_max=float(fn.test_accuracy.max()),
                        train_s=summ(fn.train_seconds), eval_s=summ(fn.eval_seconds), n_trials=len(fn),
                        same_params_synthetic=summ(fn.test_accuracy_same_params_synthetic),
                        diff_vs_primary=dict(mean=float(d.mean()), lo=lo, hi=hi, p_w=float(w.pvalue) if w else 1.0),
                        layout=fn.physical_layout.iloc[0], n_test=int(fn.n_test.iloc[0]))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.0))
    ax[0].plot(fn.seed, fn.test_accuracy, "o-", color="#2a78d6", ms=6, lw=1.2, label="FakeNairobiV2 (calibrated snapshot)")
    ax[0].plot(syn.loc[fn.seed].index, syn.loc[fn.seed].values, "s--", color="#eb6834", ms=5, lw=1, label="Primary synthetic model ($\\epsilon_{2q}=10^{-2}$)")
    a = S["nairobi"]["acc"]
    ax[0].axhline(a["mean"], color="#2a78d6", lw=1, ls=":")
    ax[0].axhspan(a["mean"] - a["sd"], a["mean"] + a["sd"], color="#2a78d6", alpha=0.08)
    ax[0].set_xlabel("Split seed"); ax[0].set_ylabel("Test accuracy"); ax[0].set_ylim(0, 1.05); ax[0].grid(True)
    ax[0].set_title(f"(a) Test accuracy, mean {a['mean']:.3f} $\\pm$ {a['sd']:.3f} (SD)"); ax[0].legend(frameon=False, loc="lower left", fontsize=8.5)
    ax[1].bar(fn.seed, fn.train_seconds, 0.6, color="#2a78d6")
    ax[1].set_xlabel("Split seed"); ax[1].set_ylabel("Local training wall-clock time (s)"); ax[1].grid(axis="y")
    ax[1].set_ylim(0, fn.train_seconds.max() * 1.2)
    ax[1].set_title(f"(b) Local training time (20 SPSA iterations)")
    fig.tight_layout(); fig.savefig(FIG / "Figure_9_Calibrated_Model.png", bbox_inches="tight"); plt.close(fig)

# ---------------------------------------------------------------- Table 11 (Pre-VDCS) and Table 12
if T:
    bc = "breast_cancer"; Aref = S["A_ref"][bc]["mean"]; T0 = min(T.values()); C0 = burden(4, 2)
    w = np.array([6, 5, 4, 2]) / 17; rows = []; S["prevdcs"] = {}
    for c in C.CONFIGS:
        A = S["vqc"][f"{bc}|{c[0]}|{c[1]}"]["mean"]
        comp = np.array([min(1, A / Aref), min(1, T0 / T[c]), fid(*c), min(1, C0 / burden(*c))])
        sc = 100 * float(w @ comp); S["prevdcs"][f"{c[0]}Q-{c[1]}L"] = dict(sA=comp[0], sT=comp[1], sN=comp[2], sC=comp[3], score=sc)
        rows.append(f"VQC ($n={c[0]}, L={c[1]}$) & {comp[0]:.4f} & {comp[1]:.4f} & {comp[2]:.4f} & {comp[3]:.4f} & \\textbf{{{sc:.2f}}} \\\\")
    S["T0"] = T0
    (OUT / "table11_prevdcs.tex").write_text("\n".join(rows) + "\n")
rows = []
for ds in C.DATASETS:
    A = S["vqc"][f"{ds}|8|4"]["mean"]; R = S["A_ref"][ds]["mean"]
    rows.append(f"{C.LABEL[ds]} & {A:.4f} & {R:.4f} & ${A - R:.4f}$ \\\\")
(OUT / "table12_breakeven.tex").write_text("\n".join(rows) + "\n")

(OUT / "summary.json").write_text(json.dumps(S, indent=2, default=lambda o: o if not isinstance(o, np.generic) else o.item()))
print(json.dumps({"A_ref": S["A_ref"], "vqc": {k: round(v["mean"], 4) for k, v in S["vqc"].items()},
                  "prevdcs": {k: round(v["score"], 2) for k, v in S.get("prevdcs", {}).items()}}, indent=1))


In [ ]:
%%writefile training_diagnostics.py
"""Optimisation diagnostics for the primary VQC run (eps_2q = 1e-2, retrained protocol).

Reports, per configuration and pooled:
  * relative reduction of the SPSA loss estimate between iteration 1 and iteration 20;
  * Pearson correlation between train and test accuracy across trials;
  * number of trials with test accuracy below 0.5 and below the majority-class rate;
  * mean test accuracy minus majority-class rate.
Usage: python training_diagnostics.py results tables
"""
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import common as C

RES = Path(sys.argv[1] if len(sys.argv) > 1 else "results")
OUT = Path(sys.argv[2] if len(sys.argv) > 2 else "tables"); OUT.mkdir(parents=True, exist_ok=True)

h = pd.read_csv(RES / "vqc_loss_history.csv"); h = h[np.isclose(h.eps_2q, 1e-2)]
v = pd.read_csv(RES / "vqc_trials.csv"); r = v[(v.protocol == "retrained") & np.isclose(v.eps_2q, 1e-2)]
keys = ["dataset", "n_qubits", "n_layers", "seed"]
g = h.sort_values("iteration").groupby(keys).loss
drop = ((g.first() - g.last()) / g.first()).rename("rel_drop").reset_index()
r = r.merge(drop, on=keys)


def majority(ds, seed):
    _, _, ytr, yte = C.matched_data(ds, seed, 4)
    maj = 1 if np.sum(ytr == 1) >= np.sum(ytr == -1) else -1
    return float(np.mean(yte == maj))


r["majority"] = [majority(d, s) for d, s in zip(r.dataset, r.seed)]
out = {"per_config": {}}
for (ds, n, L), x in r.groupby(["dataset", "n_qubits", "n_layers"]):
    out["per_config"][f"{ds}|{n}|{L}"] = dict(
        rel_loss_drop_mean=float(x.rel_drop.mean()), rel_loss_drop_min=float(x.rel_drop.min()),
        rel_loss_drop_max=float(x.rel_drop.max()),
        r_train_test=float(np.corrcoef(x.train_accuracy, x.test_accuracy)[0, 1]),
        n_below_half=int((x.test_accuracy < 0.5).sum()),
        n_below_majority=int((x.test_accuracy < x.majority).sum()),
        mean_minus_majority=float((x.test_accuracy - x.majority).mean()))
out["pooled"] = dict(
    n_trials=int(len(r)),
    rel_loss_drop_mean=float(r.rel_drop.mean()), rel_loss_drop_median=float(r.rel_drop.median()),
    r_train_test=float(np.corrcoef(r.train_accuracy, r.test_accuracy)[0, 1]),
    n_below_half=int((r.test_accuracy < 0.5).sum()),
    n_below_majority=int((r.test_accuracy < r.majority).sum()))
(OUT / "training_diagnostics.json").write_text(json.dumps(out, indent=1))
print(json.dumps(out, indent=1))


In [ ]:
%%writefile make_analytical_figures.py
#!/usr/bin/env python3
"""Regenerate the analytical figures (Figures 4, 5 and 8) from the revised circuit model.

Circuit model (Eqs. 2-5 of the manuscript, logical pre-transpilation counts):
    G_1q(n, L) = n (2L + 2)      n encoding R_Y + n initial trainable R_Y + 2nL layer R_Y/R_Z
    G_2q(n, L) = (n - 1) L       open nearest-neighbour CNOT chain per layer
    P(n, L)    = n (2L + 1)      trainable parameters
    D(n, L)   <= 2 + L (n + 1)   logical-depth upper bound
    C(n, L)    = G_1q + 2 G_2q   weighted circuit burden
    F          = (1 - eps_1q)^G_1q (1 - eps_2q)^G_2q   gate-survival proxy

Local timing values (Figure 8) are copied from Table 9 of the manuscript; they are
NOT recomputed here.

Usage:  python3 make_analytical_figures.py [output_dir]
"""
import sys
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from mpl_toolkits.mplot3d import Axes3D  # noqa: E402,F401

OUT = Path(sys.argv[1]) if len(sys.argv) > 1 else Path(".")
OUT.mkdir(parents=True, exist_ok=True)

EPS_1Q = 1e-3
EPS_2Q_GRID = [1e-3, 1e-2, 1e-1]
CONFIGS = [(4, 2), (8, 2), (8, 4)]
LABELS = {c: f"{c[0]}Q–{c[1]}L" for c in CONFIGS}
COLORS = {(4, 2): "#2a78d6", (8, 2): "#eb6834", (8, 4): "#1baf7a"}
MARKERS = {(4, 2): "o", (8, 2): "s", (8, 4): "D"}

# Table 9 (local Aer wall-clock, seconds per 20 circuits)
TIMING = {(4, 2): (0.030, 0.020), (8, 2): (0.028, 1.194), (8, 4): (0.034, 1.630)}


def g1q(n, L):
    return n * (2 * L + 2)


def g2q(n, L):
    return (n - 1) * L


def params(n, L):
    return n * (2 * L + 1)


def depth_bound(n, L):
    return 2 + L * (n + 1)


def burden(n, L):
    return g1q(n, L) + 2 * g2q(n, L)


def fidelity(n, L, e2, e1=EPS_1Q):
    return (1 - e1) ** g1q(n, L) * (1 - e2) ** g2q(n, L)


plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#555555", "axes.labelcolor": "#222222",
    "xtick.color": "#444444", "ytick.color": "#444444",
    "grid.color": "#dddddd", "grid.linewidth": 0.6,
})

# ---------------------------------------------------------------- self-check
EXPECTED = {(4, 2): (24, 6, 20, 12, 36), (8, 2): (48, 14, 40, 20, 76), (8, 4): (80, 28, 72, 38, 136)}
for c, exp in EXPECTED.items():
    got = (g1q(*c), g2q(*c), params(*c), depth_bound(*c), burden(*c))
    assert got == exp, (c, got, exp)
assert [round(100 * fidelity(*c, 1e-2), 2) for c in CONFIGS] == [91.91, 82.80, 69.67]

print("config   G1q G2q  P  D<=  C    F(1e-3)  F(1e-2)  F(1e-1)")
for c in CONFIGS:
    f = [100 * fidelity(*c, e) for e in EPS_2Q_GRID]
    print(f"{LABELS[c]:7s} {g1q(*c):4d} {g2q(*c):3d} {params(*c):3d} {depth_bound(*c):3d} {burden(*c):4d}"
          f"  {f[0]:7.2f}  {f[1]:7.2f}  {f[2]:7.2f}")

n_grid = np.arange(2, 13)
L_grid = np.arange(1, 7)
NN, LL = np.meshgrid(n_grid, L_grid)

# ---------------------------------------------------------------- Figure 4
fig = plt.figure(figsize=(12, 4.8))
ax = fig.add_subplot(1, 2, 1, projection="3d")
CC = burden(NN, LL)
ax.plot_surface(NN, LL, CC, cmap="Blues", alpha=0.55, linewidth=0.3, edgecolor="#9ec5f4",
                antialiased=True, vmin=0, vmax=CC.max() * 1.1)
for c in CONFIGS:
    ax.scatter(*c, burden(*c), s=60, color=COLORS[c], marker=MARKERS[c],
               edgecolor="white", linewidth=1.2, depthshade=False, zorder=10)
    ax.text(c[0], c[1], burden(*c) + 18, f"{LABELS[c]}\nC={burden(*c)}", fontsize=8.5,
            ha="center", color="#222222", zorder=11)
ax.set_xlabel("Qubits, $n$")
ax.set_ylabel("Layers, $L$")
ax.set_zlabel("$C=G_{1q}+2G_{2q}$")
ax.set_xticks([2, 4, 6, 8, 10, 12])
ax.set_yticks(range(1, 7))
ax.view_init(elev=24, azim=-58)
ax.set_title("(a) Weighted circuit-burden surface", pad=4)
for a in (ax.xaxis, ax.yaxis, ax.zaxis):
    a.pane.set_facecolor((0.97, 0.97, 0.97, 1.0))

ax2 = fig.add_subplot(1, 2, 2)
for c in CONFIGS:
    d = depth_bound(*c)
    ax2.scatter(g1q(*c), g2q(*c), s=18 * d, color=COLORS[c], marker=MARKERS[c],
                edgecolor="white", linewidth=1.5, zorder=3)
    ax2.annotate(f"{LABELS[c]}\n$G_{{1q}}$={g1q(*c)}, $G_{{2q}}$={g2q(*c)}\n$D\\leq{d}$",
                 (g1q(*c), g2q(*c)), xytext=(14, -4), textcoords="offset points",
                 fontsize=8.5, va="center", color="#222222")
ax2.set_xlim(10, 110)
ax2.set_ylim(0, 34)
ax2.set_xlabel("One-qubit gate count $G_{1q}=n(2L+2)$")
ax2.set_ylabel("Two-qubit gate count $G_{2q}=(n-1)L$")
ax2.grid(True)
ax2.set_title("(b) Evaluated configurations (marker area $\\propto$ depth bound)")
fig.tight_layout()
fig.savefig(OUT / "Figure_4_Complexity_3D.png", bbox_inches="tight")
plt.close(fig)

# ---------------------------------------------------------------- Figure 5
fig = plt.figure(figsize=(12, 4.8))
ax = fig.add_subplot(1, 2, 1, projection="3d")
FF = 100 * fidelity(NN, LL, 1e-2)
ax.plot_surface(NN, LL, FF, cmap="Blues_r", alpha=0.55, linewidth=0.3, edgecolor="#9ec5f4",
                vmin=FF.min() - 15, vmax=100)
for c in CONFIGS:
    f = 100 * fidelity(*c, 1e-2)
    ax.scatter(*c, f, s=60, color=COLORS[c], marker=MARKERS[c], edgecolor="white",
               linewidth=1.2, depthshade=False, zorder=10)
    ax.text(c[0] + {(4, 2): 0, (8, 2): -1.8, (8, 4): 1.8}[c], c[1], f + 6, f"{LABELS[c]}\n{f:.2f}%", fontsize=8.5, ha="center",
            color="#222222", zorder=11)
ax.set_xlabel("Qubits, $n$")
ax.set_ylabel("Layers, $L$")
ax.set_zlabel("Gate-survival proxy $F$ (%)")
ax.set_xticks([2, 4, 6, 8, 10, 12])
ax.set_yticks(range(1, 7))
ax.view_init(elev=24, azim=-58)
ax.set_title("(a) $F(n,L)$ at $\\epsilon_{1q}=10^{-3}$, $\\epsilon_{2q}=10^{-2}$", pad=4)
for a in (ax.xaxis, ax.yaxis, ax.zaxis):
    a.pane.set_facecolor((0.97, 0.97, 0.97, 1.0))

ax2 = fig.add_subplot(1, 2, 2)
for c in CONFIGS:
    f = [100 * fidelity(*c, e) for e in EPS_2Q_GRID]
    ax2.plot(EPS_2Q_GRID, f, color=COLORS[c], marker=MARKERS[c], markersize=7, linewidth=2,
             markeredgecolor="white", markeredgewidth=1.2, label=LABELS[c])
    ax2.annotate(f"{f[-1]:.1f}%", (EPS_2Q_GRID[-1], f[-1]), xytext=(7, 0),
                 textcoords="offset points", va="center", fontsize=8.5, color="#222222")
    ax2.annotate(f"{f[1]:.2f}%", (EPS_2Q_GRID[1], f[1]), xytext=(-8, -12 if c != (4, 2) else 8),
                 textcoords="offset points", ha="right", va="center", fontsize=8.5, color="#222222")
ax2.set_xscale("log")
ax2.set_xlim(7e-4, 1.8e-1)
ax2.set_ylim(0, 100)
ax2.set_xlabel("Two-qubit depolarizing error rate $\\epsilon_{2q}$ ($\\epsilon_{1q}=10^{-3}$)")
ax2.set_ylabel("Gate-survival proxy $F$ (%)")
ax2.grid(True, which="major")
ax2.legend(frameon=False, loc="lower left")
ax2.set_title("(b) Error-rate slices for evaluated configurations")
fig.tight_layout()
fig.savefig(OUT / "Figure_5_Noise_3D.png", bbox_inches="tight")
plt.close(fig)

# ---------------------------------------------------------------- Figure 8
fig, ax = plt.subplots(figsize=(6.4, 4.4))
for c in CONFIGS:
    tc, te = TIMING[c]
    tot = tc + te
    share = 100 * te / tot
    ax.scatter(burden(*c), tot, s=4 * burden(*c), color=COLORS[c], marker=MARKERS[c],
               edgecolor="white", linewidth=1.5, zorder=3)
    off = (16, -2) if c != (8, 4) else (-16, -26)
    ax.annotate(f"{LABELS[c]} (C={burden(*c)})\n{tot:.3f} s, {share:.1f}% execution", (burden(*c), tot),
                xytext=off, textcoords="offset points", fontsize=8.5,
                ha="left" if c != (8, 4) else "right", va="center", color="#222222")
ax.set_xlim(20, 155)
ax.set_ylim(-0.1, 2.0)
ax.set_xlabel("Weighted circuit burden $C=G_{1q}+2G_{2q}$")
ax.set_ylabel("Local compile + execution time (s, 20 circuits)")
ax.grid(True)
fig.tight_layout()
fig.savefig(OUT / "Figure_8_Runtime_Burden.png", bbox_inches="tight")
plt.close(fig)
print("written:", *(p.name for p in sorted(OUT.glob("Figure_[458]_*.png"))))


## 0. Validate the simulator against a Qiskit Aer density-matrix simulation (~1 min)

In [ ]:
!{PY} test_vqc_core.py

## 1. Primary VQC experiment: Tables 7–8, Figure 7 (~40 min)

In [ ]:
!{PY} -u run_vqc.py results 2>&1 | tee log_vqc.txt

## 2. Matched classical baselines: Table 6, Figure 6 (~40 min on 4 cores). Resumes if re-run.

In [ ]:
!{PY} -u run_classical.py matched results {N_JOBS} 2>&1 | tee log_matched.txt

## 3. Nested 5×5 CV: Table 5 (~50 min). Resumes if re-run.

In [ ]:
!{PY} -u run_classical.py nested results {N_JOBS} 2>&1 | tee log_nested.txt

## 4. FakeNairobiV2 calibrated-noise check: Table 10, Figure 9

In [ ]:
!{PY} -u run_aer.py nairobi results 2>&1 | tee log_nairobi.txt

## 5. Aer timing benchmark: Table 9, Figure 8. Runs last so nothing else competes for the CPU. Record the CPU model printed below for the manuscript.

In [ ]:
!lscpu | grep -E 'Model name|^CPU'
!{PY} -u run_aer.py timing results 2>&1 | tee log_timing.txt

## 6. Build all tables, paired tests and Figures 6–9, plus the analytical Figures 4–5

In [ ]:
!{PY} build_results.py results tables figures
!{PY} training_diagnostics.py results tables
!{PY} make_analytical_figures.py figures_analytical
!ls tables figures figures_analytical

## 7. Package outputs

In [ ]:
!cd /kaggle/working && zip -qr vqc_outputs.zip vqc -x 'vqc/__pycache__/*' && ls -lh vqc_outputs.zip

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('figures/*.png')): print(p); display(Image(p, width=700))